# Day 4 Tutorial：决策树与过拟合

## Goal

在同一人工划分上只改变 `max_depth`，观察树复杂度、训练拟合和验证差距。

## Setup

训练关系接近 `y=x`，但在 `x=5` 人为加入离群标签。

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor, export_text

X_train = np.arange(12, dtype=float).reshape(-1, 1)
y_train = X_train.reshape(-1).copy()
y_train[5] = 20.0
X_valid = (np.arange(11, dtype=float) + 0.5).reshape(-1, 1)
y_valid = X_valid.reshape(-1)

print('outlier:', X_train[5, 0], y_train[5])
print('shapes:', X_train.shape, y_train.shape, X_valid.shape, y_valid.shape)

outlier: 5.0 20.0
shapes: (12, 1) (12,) (11, 1) (11,)


## Steps

用同一指标循环训练四个预先固定的树深。

In [2]:
def regression_metrics(actual, predicted):
    return {
        'rmse': float(np.sqrt(mean_squared_error(actual, predicted))),
        'r2': float(r2_score(actual, predicted)),
    }

depth_values = [1, 2, 3, None]
records = []

for depth in depth_values:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_scores = regression_metrics(y_train, model.predict(X_train))
    valid_scores = regression_metrics(y_valid, model.predict(X_valid))
    records.append({
        'max_depth': 'None' if depth is None else str(depth),
        'actual_tree_depth': model.get_depth(),
        'leaf_count': model.get_n_leaves(),
        'train_rmse': train_scores['rmse'],
        'valid_rmse': valid_scores['rmse'],
        'train_r2': train_scores['r2'],
        'valid_r2': valid_scores['r2'],
        'r2_gap': train_scores['r2'] - valid_scores['r2'],
    })

results = pd.DataFrame(records)
results

,max_depth,actual_tree_depth,leaf_count,train_rmse,valid_rmse,train_r2,valid_r2,r2_gap
0,1,1,2,3.426090,2.262360,0.578587,0.488173,0.090415
1,2,2,4,1.290994,4.512609,0.940165,-1.036364,1.976528
2,3,3,7,0.612372,4.423594,0.986537,-0.956818,1.943355
3,None,5,12,0.000000,4.397830,1.000000,-0.934091,1.934091


In [3]:
small_tree = DecisionTreeRegressor(max_depth=2, random_state=42)
small_tree.fit(X_train, y_train)
print(export_text(small_tree, feature_names=['x']))

|--- x <= 4.50
|   |--- x <= 1.50
|   |   |--- value: [0.50]
|   |--- x >  1.50
|   |   |--- value: [3.00]
|--- x >  4.50
|   |--- x <= 5.50
|   |   |--- value: [20.00]
|   |--- x >  5.50
|   |   |--- value: [8.50]



## Checks

不限深树应能记住这份训练数据；这不代表它能泛化。

In [4]:
unrestricted = DecisionTreeRegressor(max_depth=None, random_state=42)
unrestricted.fit(X_train, y_train)
assert np.allclose(unrestricted.predict(X_train), y_train)
assert results['actual_tree_depth'].is_monotonic_increasing
assert np.isfinite(results.select_dtypes('number').to_numpy()).all()
print('tree complexity checks passed')

tree complexity checks passed


## Next Steps

完成 `03_exercises.md`，并在个人副本中先写深度趋势预判，再运行和解释；不要用 test 选择树深。